# Pipeline da Entrega 1 - pre-treino

Estado final do projeto: modelo LLaMA-inspired de ~109M parametros, corpus FineWeb-Edu `sample-10BT`, tokenizer GPT-2 BPE e treino local na RTX 3060 ate 60k steps.

Arquivos principais:

- config oficial: `configs/pretrain_llama_100m.yaml`;
- config local final: `configs/pretrain_llama_100m_rtx3060_60k.yaml`;
- log final: `outputs/train_log_0_60k.csv`;
- curva final: `outputs/loss_curve_0_60k.png`;
- samples finais: `outputs/samples_best_60k.txt`;
- melhor checkpoint: `checkpoints/ckpt_best.pt`;
- ultimo checkpoint: `checkpoints/ckpt_last.pt`.

## 1. Ambiente

Use o kernel `Python 3.11 (Prof2 CUDA)`. A celula abaixo deve mostrar `cuda disponivel: True`.

In [ ]:
import sys
import torch

print('python:', sys.executable)
print('torch:', torch.__version__)
print('cuda disponivel:', torch.cuda.is_available())
print('gpus:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## 2. Dados

Corpus local final:

- `data/fineweb_edu_train_500m.bin`: 512.824.558 tokens de treino;
- `data/fineweb_edu_val_500m.bin`: 5.133.236 tokens de validacao.

In [ ]:
from pathlib import Path
import numpy as np

paths = [
    ('train', Path('data/fineweb_edu_train_500m.bin')),
    ('val', Path('data/fineweb_edu_val_500m.bin')),
]

for label, path in paths:
    if not path.exists():
        print(label, 'NAO ENCONTRADO:', path)
        continue
    tokens = np.memmap(path, dtype=np.uint16, mode='r')
    mb = path.stat().st_size / 1024 / 1024
    print(f'{label}: {len(tokens):,} tokens | {mb:.2f} MB | {path}')

## 3. Preparacao de dados

Use apenas se precisar recriar o corpus binario.

In [ ]:
# !python scripts/prepare_data.py --config configs/pretrain_llama_100m.yaml --max-documents 500000

## 4. Smoke test

Use para validar rapidamente tokenizer, dataset, forward, backward, checkpoint e geracao com um modelo pequeno.

In [ ]:
# !python scripts/run_smoke_test.py

## 5. Treino final local

O treino final foi mantido em 60k steps. Se `checkpoints/ckpt_last.pt` ja estiver no step 60.000, esta celula nao precisa ser rodada de novo.

In [ ]:
# Treino final local na RTX 3060 ate 60k steps.
!python src/train.py --config configs/pretrain_llama_100m_rtx3060_60k.yaml --resume checkpoints/ckpt_last.pt

## 6. Resultados

Gera a curva final a partir do log unificado 0-60k.

In [ ]:
!python src/plot_loss.py --log outputs/train_log_0_60k.csv --output outputs/loss_curve_0_60k.png

## 7. Geracao

Gera exemplos usando o melhor checkpoint observado na validacao.

In [ ]:
!python src/generate.py --checkpoint checkpoints/ckpt_best.pt --prompt 'The future of artificial intelligence is' --max_new_tokens 100 --temperature 0.8 --top_k 50 --output outputs/samples_best_60k.txt
!python src/generate.py --checkpoint checkpoints/ckpt_best.pt --prompt 'Education is important because' --max_new_tokens 100 --temperature 0.8 --top_k 50 --output outputs/samples_best_60k.txt
!python src/generate.py --checkpoint checkpoints/ckpt_best.pt --prompt 'Machine learning is a field of study that' --max_new_tokens 100 --temperature 0.8 --top_k 50 --output outputs/samples_best_60k.txt